In [14]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [15]:
current_dir = Path.cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done")

In [16]:
# let's make a mixture of 1d normal and 1d exponential distribution
MEAN=30
STD=10
SIZE=100
EXP_SCALE=10 
PRIOR=0.5
data = pd.DataFrame()
data["normal"] = np.random.normal(loc=MEAN, scale=STD, size=SIZE)
data["exp"] = np.random.exponential(scale=EXP_SCALE,size=SIZE)

In [17]:
data.head()

,normal,exp
0,13.841593,5.794711
1,29.672613,4.133922
2,12.802738,1.743308
3,38.857828,12.011765
4,29.291156,2.086440


In [18]:
def normal_pdf(x:float, mean:float, var:float) -> float:
    num = np.exp(-0.5*((x-mean)**2/(var + 1e-15)))
    deno = np.sqrt(2*np.pi*var)
    return num/deno

In [19]:
def exponential_pdf(x, scale):
    scale = max(scale, 1e-15) 
    pdf_values = np.zeros_like(x, dtype=float)
    mask = x > 0
    pdf_values[mask] = (1.0 / scale) * np.exp(-x[mask] / scale)
    return pdf_values + 1e-15

In [20]:
exponential_pdf(x=data['normal'], scale=EXP_SCALE)

array([0.02505343, 0.0051444 , 0.02779612, 0.00205317, 0.00534443,
       0.00237403, 0.00228873, 0.01003896, 0.00357441, 0.0047841 ,
       0.00433332, 0.00324551, 0.02514656, 0.0007691 , 0.00768741,
       0.00570478, 0.0060729 , 0.01139684, 0.00257427, 0.00179641,
       0.00171993, 0.00205211, 0.00102961, 0.00600285, 0.02352224,
       0.0068341 , 0.00221845, 0.00882966, 0.0081944 , 0.00452754,
       0.00088327, 0.0171258 , 0.03094294, 0.00158738, 0.00573819,
       0.00476638, 0.00747318, 0.00566105, 0.00970116, 0.00730751,
       0.00473785, 0.00856346, 0.00410646, 0.00156516, 0.02043353,
       0.0036337 , 0.00585789, 0.00424698, 0.00222895, 0.01315746,
       0.01118724, 0.00769576, 0.00851841, 0.02020968, 0.00089795,
       0.00341681, 0.00059237, 0.00101489, 0.00119508, 0.00165237,
       0.00755841, 0.03964181, 0.00273935, 0.00041896, 0.0020189 ,
       0.00632172, 0.00424393, 0.00182886, 0.00531201, 0.00698349,
       0.00173116, 0.04583016, 0.00471787, 0.0061415 , 0.00452

In [21]:
def multimodel_single_dim(x: pd.DataFrame, prior: float, iterations: int = 10):
    # convergence data
    converge_df = []
    
    # random initialization
    initial_mean = np.mean(x)
    initial_std = np.std(x)
    initial_scale = np.random.uniform(size=1)
    
    for _ in range(iterations):
        
        # calculating likelihood
        normal_likelihoods = normal_pdf(x, mean=initial_mean, var=initial_std**2)
        exp_likelihoods = exponential_pdf(x, scale=initial_scale)
        # print(f"normal_likelihoods: {normal_likelihoods}")
        # print(f"exp_likelihoods: {exp_likelihoods}")
        
        # calculating the posteriors or responsibilities
        b = (normal_likelihoods * prior)/(((normal_likelihoods * prior) + (exp_likelihoods * (1-prior))) + 1e-15)
        a = 1 - b
        # print(f"normal_res: {b}")
        # print(f"exp_res: {a}")
        
        # calculating the updated parameters
        updated_mean = (b * x).sum()/b.sum()
        updated_std = np.sqrt((((x - updated_mean)**2)*b).sum()/b.sum() + 1e-15).item()
        updated_scale = (a * x).sum()/a.sum()
        
        # updating prior
        prior = b.sum()/len(x)
        # print(f"updated_mean: {updated_mean}")
        # print(f"updated_var: {updated_var}")
        # print(f"updated_scale: {updated_scale}")
        
        # updating the parameters
        initial_mean = updated_mean
        initial_std = updated_std
        initial_scale = updated_scale
        
        # storing the data
        ite_data = {
            "updated_mean" : updated_mean,
            "updated_var" : updated_std,
            "updated_scale": updated_scale
        }
        
        converge_df.append(ite_data)
    
    return initial_mean, initial_std, initial_scale, pd.DataFrame(converge_df)

In [22]:
actual_data = np.concat([data["normal"], data["exp"]])
np.random.shuffle(actual_data)

In [23]:
# multiple_init = []
# for i in range(10):
#     final_mean, final_var, final_scale, conv_df = multimodel_single_dim(x=actual_data, prior=1/(len(data.columns)))
#     ite_data = {
#         "final_mean": final_mean,
#         "final_var": final_var,
#         "final_scale": final_scale
#     }
#     multiple_init.append(ite_data)

In [24]:
final_mean, final_var, final_scale, conv_df = multimodel_single_dim(x=actual_data, prior=1/(len(data.columns)), iterations=100)

In [25]:
print(f"Actual_mean: {MEAN}\nActual_var: {STD}\nActual_scale: {EXP_SCALE}")

Actual_mean: 30
Actual_var: 10
Actual_scale: 10


In [26]:
conv_df

,updated_mean,updated_var,updated_scale
0,23.136579,13.297027,1.687441
1,23.434654,13.271271,2.292546
2,24.050283,13.083005,2.816991
3,24.716424,12.853080,3.282537
4,25.347532,12.620079,3.698545
...,...,...,...
95,30.910390,10.106195,8.529540
96,30.910681,10.106035,8.529930
97,30.910952,10.105886,8.530292
98,30.911202,10.105749,8.530627
